# Part 3 — Full PointNet adaptation for ECG point sets

This notebook trains a fuller PointNet-based classifier with input transform, feature transform, and transform regularization, adapted to the ECG point-set representation.

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:/Users/Sherif/Desktop/ECG analysis")
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import src.train_newpointnet
import src.pointcloud_ecg
import models.pointnet_full_ecg

importlib.reload(src.train_newpointnet)
importlib.reload(src.pointcloud_ecg)
importlib.reload(src.pointnet_full_ecg)

import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from models.UNet1D import UNet1D
from models.pointnet_full_ecg import PointNetECGFull
from src.pointcloud_ecg import create_point_cloud
from src.dataset import create_dataloaders
from src.train_newpointnet import train_newpointnet, test_newpointnet


AttributeError: module 'src' has no attribute 'pointnet_full_ecg'

In [ ]:
def set_seed(seed=1913):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1913)


In [ ]:
train_loader, val_loader, test_loader, label_cols, _ = create_dataloaders()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(label_cols)
print(device)


## Load pretrained U-Net

In [ ]:
unet = torch.load("unetecg.pth", map_location=device, weights_only=False)
unet.to(device)
unet.eval()
print("Embedding dim:", unet.embedding_dim)


## Quick sanity check on point-set construction

In [ ]:
signals, labels = next(iter(train_loader))
signals = signals.to(device)

with torch.no_grad():
    emb_versions = []
    for i in range(12):
        input_lead = signals[:, i:i+1, :]          # (B, 1, T)
        _, emb_i = unet(input_lead)                # (B, 12, 32)
        emb_versions.append(emb_i)

    emb_versions = torch.stack(emb_versions, dim=0)  # (12, B, 12, 32)
    embeddings = emb_versions.mean(dim=0)            # (B, 12, 32)

print("Embeddings shape:", embeddings.shape)
point_cloud = create_point_cloud(signals, embeddings)
print("Point cloud shape:", point_cloud.shape)


## Train the full PointNet adaptation

In [ ]:
model, history = train_newpointnet(
    unet=unet,
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=len(label_cols),
    epochs=15,
    lr=1e-3,
    reg_weight=1e-3,
    device=device,
)


## Plot training history

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["train_loss"], label="Train loss")
plt.plot(history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Full PointNet training curves")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history["val_auc"], label="Val macro-AUC")
plt.xlabel("Epoch")
plt.ylabel("Macro-AUC")
plt.title("Validation macro-AUC")
plt.legend()
plt.grid(True)
plt.show()


## Test evaluation

In [ ]:
y_true, y_pred, macro_auc = test_newpointnet(
    model=model,
    unet=unet,
    test_loader=test_loader,
    label_cols=label_cols,
    reg_weight=1e-3,
    device=device,
)


## Save model

In [ ]:
torch.save(model.state_dict(), "best_full_pointnet_ecg.pt")
print("Saved to best_full_pointnet_ecg.pt")
